# 02 - Data Analysis Report (all 14 B tasks)
Every task needs a number/table/chart **and an interpretation**. The complete written report is `reports/B_analysis_report.md`.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
SEED = 42
df=pd.read_csv(ROOT/'data/processed/master_train.csv')
df['revenue_per_ha']=df.yield_tons_per_ha*10*df.price_birr_per_quintal

## B1.1 Revenue per hectare by crop

In [2]:
df.groupby('crop_type').agg(mean_yield=('yield_tons_per_ha','mean'),mean_revenue=('revenue_per_ha','mean')).round(2).sort_values('mean_revenue',ascending=False)

,mean_yield,mean_revenue
crop_type,,
teff,2.00,153349.19
wheat,2.80,135499.52
maize,3.90,114496.76
barley,2.47,110195.88
sorghum,2.61,83292.03


## B1.2 Price trend

In [3]:
p=pd.read_csv(ROOT/'data/processed/clean_prices.csv'); q=p.groupby(['crop_type','year']).price_birr_per_quintal.mean().unstack(); out=pd.DataFrame({'2021':q[2021],'2024':q[2024]}); out['change']=out['2024']-out['2021']; out['pct_change']=100*out.change/out['2021']; out.round(2)

,2021,2024,change,pct_change
crop_type,,,,
barley,3946.8,4977.8,1031.0,26.12
maize,2602.2,3233.2,631.0,24.25
sorghum,2688.8,3492.4,803.6,29.89
teff,6885.0,8575.3,1690.3,24.55
wheat,4305.0,5557.4,1252.4,29.09


## B2.1-B2.3 Region and crop

In [4]:
display(df.groupby('region').yield_tons_per_ha.agg(['mean','median','std']).round(3))
display(df.groupby('crop_type').yield_tons_per_ha.agg(['mean','median','std']).round(3))
display(df.pivot_table(index='region',columns='crop_type',values='yield_tons_per_ha').round(3))

,mean,median,std
region,,,
Amhara,3.129,3.021,1.231
Oromia,3.073,2.893,1.292
SNNPR,3.041,2.741,1.472
Somali,1.464,1.294,0.771
Tigray,3.131,2.932,1.309


,mean,median,std
crop_type,,,
barley,2.467,2.376,1.158
maize,3.904,3.781,1.584
sorghum,2.611,2.514,0.974
teff,2.003,2.001,0.988
wheat,2.802,2.692,1.411


crop_type,barley,maize,sorghum,teff,wheat
region,,,,,
Amhara,3.187,3.829,2.495,2.479,3.598
Oromia,2.675,4.450,2.955,2.288,3.120
SNNPR,2.289,4.797,3.106,2.104,2.668
Somali,1.123,2.320,1.793,0.831,1.281
Tigray,3.010,4.074,2.709,2.332,3.463


## B2.4-B2.5 Improved seed and pest/disease

In [5]:
for col in ['improved_seed_used','pest_disease_flag']:
    t=df.dropna(subset=[col]).groupby(['crop_type',col]).yield_tons_per_ha.mean().unstack(); t['gap']=t.get(1)-t.get(0); t['pct_gap']=100*t.gap/t.get(0); display(t.round(3))

improved_seed_used,0,1,gap,pct_gap
crop_type,,,,
barley,2.305,2.727,0.422,18.296
maize,3.612,4.394,0.783,21.667
sorghum,2.442,2.889,0.447,18.309
teff,1.844,2.263,0.419,22.746
wheat,2.596,3.133,0.537,20.665


pest_disease_flag,0.0,1.0,gap,pct_gap
crop_type,,,,
barley,2.624,1.917,-0.706,-26.922
maize,4.187,2.972,-1.215,-29.029
sorghum,2.771,2.026,-0.745,-26.896
teff,2.130,1.568,-0.562,-26.366
wheat,2.980,2.191,-0.788,-26.456


## B3.1-B3.4 Agronomic drivers

In [6]:
d=df.copy(); d['fert_band']=pd.qcut(d.fertilizer_kg_per_ha,4,duplicates='drop'); d['alt_band']=pd.cut(d.altitude_m,4); d['dist_band']=pd.qcut(d.distance_to_market_km,4,duplicates='drop')
display(d.groupby('fert_band',observed=True).yield_tons_per_ha.mean().round(3))
display(d.groupby(['crop_type','alt_band'],observed=True).yield_tons_per_ha.agg(['mean','count']).round(3))
display(d.pivot_table(index='planting_month',columns='crop_type',values='yield_tons_per_ha').round(3))
print('Distance correlation:',d[['distance_to_market_km','yield_tons_per_ha']].corr().iloc[0,1].round(3)); display(d.groupby('dist_band',observed=True).yield_tons_per_ha.mean().round(3))

fert_band
(0.497, 19.707]      2.427
(19.707, 33.827]     2.676
(33.827, 53.225]     2.856
(53.225, 898.848]    3.096
Name: yield_tons_per_ha, dtype: float64

mean  count
crop_type alt_band                          
barley    (52.76, 816.853]      2.871     10
          (816.853, 1577.902]   0.205      4
          (1577.902, 2338.951]  1.618    442
          (2338.951, 3100.0]    2.620   2478
maize     (52.76, 816.853]      2.346    152
          (816.853, 1577.902]   4.190   2025
          (1577.902, 2338.951]  3.542    874
          (2338.951, 3100.0]    0.713      8
sorghum   (52.76, 816.853]      1.828    252
          (816.853, 1577.902]   2.806   2145
          (1577.902, 2338.951]  2.268    602
          (2338.951, 3100.0]    0.662      7
teff      (52.76, 816.853]      2.504     14
          (816.853, 1577.902]   0.985    237
          (1577.902, 2338.951]  2.250   2041
          (2338.951, 3100.0]    1.627    723
wheat     (52.76, 816.853]      3.139     11
          (816.853, 1577.902]   0.135     11
          (1577.902, 2338.951]  2.083    707
          (2338.951, 3100.0]    3.029   2347

crop_type,barley,maize,sorghum,teff,wheat
planting_month,,,,,
Aug,2.576,3.875,2.618,2.020,2.805
Feb,2.241,4.118,2.729,1.852,2.495
Jul,2.573,3.846,2.526,2.107,2.932
Jun,2.657,3.725,2.475,2.071,3.078
Mar,2.313,3.956,2.701,1.960,2.687


Distance correlation: 0.005


dist_band
(0.499, 3.435]    2.769
(3.435, 8.232]    2.730
(8.232, 16.43]    2.775
(16.43, 90.0]     2.777
Name: yield_tons_per_ha, dtype: float64

## B4.1-B4.3 Time and weather

In [7]:
display(df.pivot_table(index='survey_year',columns='region',values='yield_tons_per_ha').round(3))
ry=df.groupby(['region','survey_year']).agg(temp=('season_avg_temp_c','mean'),yield_mean=('yield_tons_per_ha','mean')).reset_index(); ry['region_4yr_temp']=ry.groupby('region').temp.transform('mean'); ry['temp_anomaly']=ry.temp-ry.region_4yr_temp; display(ry.round(3))
r=df[['rainfall_mm_season','season_rainfall_mm']].dropna(); print('Rainfall correlation:',r.corr().iloc[0,1].round(3)); print('Mean absolute difference:',(r.iloc[:,0]-r.iloc[:,1]).abs().mean().round(2),'mm')

region,Amhara,Oromia,SNNPR,Somali,Tigray
survey_year,,,,,
2021,3.146,3.041,2.864,1.283,3.159
2022,3.080,3.031,3.024,1.671,3.139
2023,3.105,3.132,3.161,1.415,3.155
2024,3.181,3.089,3.105,1.478,3.068


,region,survey_year,temp,yield_mean,region_4yr_temp,temp_anomaly
0,Amhara,2021,16.736,3.146,16.785,-0.049
1,Amhara,2022,18.312,3.080,16.785,1.528
2,Amhara,2023,15.506,3.105,16.785,-1.279
3,Amhara,2024,16.585,3.181,16.785,-0.200
4,Oromia,2021,19.131,3.041,18.864,0.267
5,Oromia,2022,19.715,3.031,18.864,0.852
6,Oromia,2023,18.505,3.132,18.864,-0.359
7,Oromia,2024,18.104,3.089,18.864,-0.760
8,SNNPR,2021,21.389,2.864,20.146,1.243
9,SNNPR,2022,19.713,3.024,20.146,-0.433


Rainfall correlation: 0.007
Mean absolute difference: 501.36 mm


## Read the final interpretations
Open `reports/B_analysis_report.md`. It contains all B1.1-B4.3 headings and the required one/two-sentence interpretations.